In [1]:
import os
import io
import torch
import base64
from tqdm import tqdm
from PIL import Image
from functools import lru_cache
from typing import List, Dict, Any, Iterator

from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from huggingface_hub import InferenceClient
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

In [2]:
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN 
    

In [3]:
from transformers.models.qwen2_vl.modeling_qwen2_vl import Qwen2VLForConditionalGeneration


@lru_cache(maxsize=1)
def load_models(
    whisper_model_size: str = "medium",
    t5_model_path: str = "../models/rut5-cleaner-tuned/",
    e5_linker_path: str = "../models/e5-linker-tuned/",
    vlm_model_name: str = "Qwen/Qwen2-VL-7B-Instruct",
    hf_token: str | None = None
) -> Dict[str, Any]:
    print("Loading models into memory...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"

    whisper_model = WhisperModel(whisper_model_size, device=device, compute_type=compute_type)

    t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_path, token=hf_token)
    t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_path, token=hf_token).to(device)
    t5_model.eval()

    e5_linker = SentenceTransformer(e5_linker_path, token=hf_token, device=device)

    vlm_model: Qwen2VLForConditionalGeneration = Qwen2VLForConditionalGeneration.from_pretrained(
        vlm_model_name,
        torch_dtype=torch.bfloat16,
        device_map="cuda:0", 
        token=hf_token
    )
    vlm_processor = AutoProcessor.from_pretrained(vlm_model_name, token=hf_token)
    vlm_model.eval()
    
    return {
        "whisper": whisper_model,
        "t5_tokenizer": t5_tokenizer,
        "t5_model": t5_model,
        "e5_linker": e5_linker,
        "vlm_model": vlm_model,
        "vlm_processor": vlm_processor,
    }

In [4]:
def transcribe_audio(audio_path: str, whisper_model: WhisperModel) -> Iterator[Dict[str, Any]]:
    print(f"Transcribing {audio_path}...")
    segments, _ = whisper_model.transcribe(audio_path, beam_size=5, language="ru", vad_filter=True)
    
    for segment in tqdm(segments):
        yield {
            "start": segment.start,
            "end": segment.end,
            "text": segment.text
        }

def group_segments(segments: Iterator[Dict], max_chars: int = 700) -> Iterator[str]:
    current_chunk = ""
    sentence_endings = ('.', '!', '?', '...')
    
    for seg in segments:
        text = seg['text'].strip()
        if not text: continue
        
        current_chunk += " " + text
        if len(current_chunk) >= max_chars and text.endswith(sentence_endings):
            yield current_chunk.strip()
            current_chunk = ""
            
    if current_chunk.strip():
        yield current_chunk.strip()

In [22]:
class SequentialLinker:
    def __init__(
        self, 
        model, 
        materials: List[Dict], 
        lookahead_window: int = 3,       
        confidence_threshold: float = 0.40,
        step_penalty: float = 0.02,       
        switch_artifact_penalty: float = 0.1,
    ):
        self.model = model
        
        self.lookahead_window = lookahead_window
        self.confidence_threshold = confidence_threshold
        self.step_penalty = step_penalty
        self.switch_artifact_penalty = switch_artifact_penalty
        
        self.artifacts: dict[str, list[dict]] = {}
        for m in materials:
            art_name = self._get_artifact_name(m["id"])
            if art_name not in self.artifacts:
                self.artifacts[art_name] = []
            self.artifacts[art_name].append(m)
            
        self.state = {}
        self.embeddings = {}
        self.last_active_artifact = None 
        
        print(f"Encoding {len(materials)} materials across {len(self.artifacts)} files...")
        for art_name, mats in self.artifacts.items():
            self.state[art_name] = 0
            
            formatted_passages = ["passage: " + m["content"] for m in mats]
            self.embeddings[art_name] = self.model.encode(formatted_passages, convert_to_tensor=True)

    def _get_artifact_name(self, doc_id: str) -> str:
        """Extracts the base filename from the ID generated by the parsers."""
        if "_Slide_" in doc_id:
            return doc_id.split("_Slide_")[0]
        elif "_block_" in doc_id:
            return doc_id.split("_block_")[0]
        return "unknown_artifact"
        
    def predict(self, spoken_text: str):
        query_emb = self.model.encode("query: " + spoken_text, convert_to_tensor=True)
        
        global_best_score = -999.0
        winning_artifact = None
        winning_local_idx = 0
        
        for art_name, embs in self.embeddings.items():
            curr_idx = self.state[art_name]
            
            if curr_idx >= len(embs):
                continue
                
            end_idx = min(curr_idx + self.lookahead_window + 1, len(embs))
            window_embs = embs[curr_idx:end_idx]
            
            scores = util.cos_sim(query_emb, window_embs)[0] 
            for i in range(len(scores)):
                scores[i] -= (i * self.step_penalty)
                
            if self.last_active_artifact and self.last_active_artifact != art_name:
                scores -= self.switch_artifact_penalty
                
            local_best_offset = int(torch.argmax(scores).item())
            local_best_score = scores[local_best_offset].item()
            
            if local_best_score > global_best_score:
                global_best_score = local_best_score
                winning_artifact = art_name
                winning_local_idx = curr_idx + local_best_offset
                
        if winning_artifact and global_best_score > self.confidence_threshold:
            self.state[winning_artifact] = winning_local_idx
            self.last_active_artifact = winning_artifact
            
        if not winning_artifact:
            return {"matched_id": None, "matched_content": None, "score": 0.0}
            
        matched_material = self.artifacts[winning_artifact][winning_local_idx]
        
        raw_score = util.cos_sim(query_emb, self.embeddings[winning_artifact][winning_local_idx])[0][0].item()
        
        return {
            "matched_id": matched_material["id"],
            "matched_content": matched_material["content"],
            "score": round(raw_score, 4),
            "penalty_score": round(global_best_score, 4),
            "source_file": winning_artifact
        }

In [23]:
def clean_text_chunks(chunks: Iterator[str], model, tokenizer) -> Iterator[str]:
    print(f"Cleaning text chunks...")
    device = model.device
    
    for text in chunks:
        input_text = "clean: " + text
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=512,
                num_beams=5,  
                repetition_penalty=2.5,
                length_penalty=1.0,
                early_stopping=True
            )
        yield tokenizer.decode(outputs[0], skip_special_tokens=True)

In [24]:
def parse_pdf_to_slides_with_vlm(pdf_path: str, vlm_model: Any, vlm_processor: Any) -> List[Dict]:
    print(f"Parsing PDF and extracting visual descriptions for {pdf_path} locally...")
    
    slides = []
    doc = fitz.open(pdf_path)
    
    for i, page in enumerate(tqdm(doc)):
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_data = pix.tobytes("png")
        image = Image.open(io.BytesIO(img_data))
        
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image,
                    },
                    {
                        "type": "text", 
                        "text": "Опиши содержание сайда во всех деталях. Объясни код, формулы, таблицы и схемы. Опиши что изображено на картинках."
                    },
                ],
            }
        ]

        text = vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = vlm_processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to(vlm_model.device)

        with torch.no_grad():
            generated_ids = vlm_model.generate(**inputs, max_new_tokens=500)
            generated_ids_trimmed = [
                out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            vlm_description = vlm_processor.batch_decode(
                generated_ids_trimmed, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=False
            )[0]

        slides.append({
            "id": f"{os.path.basename(pdf_path)}_Slide_{i+1}", 
            "content": vlm_description
        })
            
    doc.close()
    return slides

In [25]:
def parse_code_to_blocks(file_path: str) -> List[Dict]:
    """Parses source code files using Langchain language-aware text splitters."""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    ext = file_path.split('.')[-1].lower()
    
    lang_map = {
        'py': Language.PYTHON, 'cpp': Language.CPP, 'c': Language.C,
        'h': Language.CPP, 'hpp': Language.CPP, 'cu': Language.CPP,
        'cuh': Language.CPP, 'java': Language.JAVA, 'go': Language.GO,
        'rs': Language.RUST,
    }
    
    lang = lang_map.get(ext)
    
    if lang:
        splitter = RecursiveCharacterTextSplitter.from_language(
            language=lang, chunk_size=1500, chunk_overlap=150
        )
    else:
        splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
        
    blocks = splitter.split_text(content)
        
    formatted_blocks = []
    for i, b in enumerate(blocks):
        if len(b.strip()) > 20:
            formatted_blocks.append({
                "id": f"{os.path.basename(file_path)}_block_{i+1}",
                "content": b.strip()
            })
            
    return formatted_blocks

In [26]:
def parse_materials(file_paths: List[str],  vlm_model: Any, vlm_processor: Any) -> List[Dict]:
    all_materials = []
    for path in file_paths:
        if path.lower().endswith(".pdf"):
            all_materials.extend(parse_pdf_to_slides_with_vlm(path,  vlm_model, vlm_processor))
        else:
            all_materials.extend(parse_code_to_blocks(path))
    return all_materials

In [27]:
def link_speech_to_materials(
    speech_chunks: Iterator[str], 
    materials: List[Dict], 
    linker_model: SentenceTransformer
) -> Iterator[Dict[str, Any]]:
    if not materials:
        print("No materials provided to link against.")
        for chunk in speech_chunks:
            yield {"speech": chunk, "matched_material_id": None, "score": 0.0}
            return

    print("Linking speech to materials...")
    linker = SequentialLinker(linker_model, materials)
    
    for chunk in speech_chunks:
        match_info = linker.predict(chunk)
        
        yield {
            "speech_summary": chunk,
            "matched_material_id": match_info["matched_id"],
            "matched_material_content": match_info["matched_content"],
            "similarity_score": match_info["score"],
            "similarity_penalty": match_info["penalty_score"],
        }

In [28]:
def process_lecture_files(file_list: List[str]) -> Iterator[Dict]:
    audio_files = [f for f in file_list if f.lower().endswith(('.mp3', '.wav', '.m4a'))]
    material_files = [f for f in file_list if not f.lower().endswith(('.mp3', '.wav', '.m4a', '.json', '.csv'))]
    
    if not audio_files:
        raise ValueError("No audio file found in the input list.")
    if len(audio_files) > 1:
        print("Multiple audio files found. Processing only the first one.")
        
    audio_path = audio_files[0]
    
    models = load_models()
    
    materials = parse_materials(material_files, models["vlm_model"], models["vlm_processor"])
    
    raw_segments = transcribe_audio(audio_path, models["whisper"])
    raw_chunks = group_segments(raw_segments)
    
    cleaned_speech = clean_text_chunks(raw_chunks, models["t5_model"], models["t5_tokenizer"])
    
    final_output = link_speech_to_materials(cleaned_speech, materials, models["e5_linker"])
    
    print("Pipeline finished successfully!")
    return final_output


In [29]:
import glob

input_files = glob.glob("../data/raw/01/*")

# try:
results = process_lecture_files(input_files)

for idx, item in enumerate(results):
    print(f"\n--- Chunk {idx + 1} ---")
    print(f"Speech : {item['speech_summary']}")
    print(f"Slide/Code ID : {item['matched_material_id']}")
    print(f"Score:   {item['similarity_score']}")
    print(f"Penalty: {item['similarity_penalty']}")
        
# except Exception as e:
#     print(f"Pipeline failed: {e}")

Parsing PDF and extracting visual descriptions for ../data/raw/01/cuda-02-sms.pdf locally...


  0%|          | 0/25 [00:00<?, ?it/s]

100%|██████████| 25/25 [03:58<00:00,  9.54s/it]


Pipeline finished successfully!
Linking speech to materials...
Encoding 35 materials across 4 files...
Cleaning text chunks...
Transcribing ../data/raw/01/2026-02-27 Лекция - Никита Шаповалов - CUDA- блоки и вычислители.mp3...


10it [00:04,  1.76it/s]

...


--- Chunk 25 ---
Speech : Надо понимать, что у нас есть Shared Memory. По сути это Instruction Cache, а L1 Cache общий Shared Memory. Shared Memory — это общий Shared Memory, который соседствует с Shared Memory на один вычислительный Unit. На одной смке 128 потоков. Выполнение может быть больше физически, но одновременно исполняться может больше потоков. На схеме CUDA Core для FP16 нет, так как один CUDA Core для FP32 не выполняется на FP32 Core, но с пропускной способностью два раза больше. Tensor Core представляет собой отдельную тему, включая разные типы поддержанных операций.
Slide/Code ID : cuda-02-sms.pdf_Slide_6
Score:   0.1201
Penalty: 0.0801


442it [02:23,  1.89it/s]


--- Chunk 26 ---
Speech : В целом, SM-ки представляют собой отдельные вычислительные юниты, не зависящие друг от друга. Существует trade-off: чем больше блоков на одной SM, тем больший размер одного вычислительного юнита. Например, если один блок запускается на одной SM, тем меньший размер.
Slide/Code ID : cuda-02-sms.pdf_Slide_4
Score:   -0.0377
Penalty: -0.0377


444it [02:23,  3.09it/s]


KeyboardInterrupt: 